# Advanced Tool Implementation & Integration
Comprehensive testing and multimodal orchestration of cooking tools.

In [1]:
import sys
import os
import asyncio
import google.generativeai as genai
from dotenv import load_dotenv

# Add root to path
sys.path.append(os.path.abspath('..'))
from app.tools import cooking_tools

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel('gemini-1.5-flash')

C:\Users\thoma\Downloads\kitchen-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\thoma\AppData\Local\Temp\ipykernel_24900\1101395329.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1. Complex Tool Scenarios
Testing edge cases for timers and conversions.

In [2]:
async def test_complex_scenarios():
    # Negative conversion
    print("Test 1 (Unsupported Unit):", await cooking_tools.convert_units(10, "lightyear", "ml"))
    
    # Overlapping timers
    await cooking_tools.set_kitchen_timer("session-1", 600, "Oven")
    await cooking_tools.set_kitchen_timer("session-1", 120, "Microwave")
    
    # Large scale factor
    print("Test 2 (Scaling x10):", await cooking_tools.scale_recipe("session-1", 10.0))

await test_complex_scenarios()

Test 1 (Unsupported Unit): {'error': 'Conversion from lightyear to ml not supported.'}
Test 2 (Scaling x10): {'status': 'success', 'new_multiplier': 10.0}


## 2. Vision-to-Tool Orchestration
Using Gemini Vision to automatically call a tool (e.g., setting a timer from an image).

In [3]:
import PIL.Image

async def vision_orchestrator(image_path):
    img = PIL.Image.open(image_path)
    prompt = "Analyze this image of a recipe. Based on the instructions, what timer should I set? Return only 'seconds:label'."
    response = model.generate_content([prompt, img])
    
    # Mocking the extraction logic
    # In real use, Gemini would call the tool via function calling
    parts = response.text.strip().split(":")
    if len(parts) == 2:
        seconds = int(parts[0])
        label = parts[1]
        return await cooking_tools.set_kitchen_timer("vision-session", seconds, label)
    
    return "No timer found in image."

print("Vision Orchestrator: [Placeholder for actual image test]")

Vision Orchestrator: [Placeholder for actual image test]


## 3. Tool Error Handling
Simulating failures to ensure robust user feedback.

In [4]:
async def test_error_handling():
    # Invalid step navigation
    res = await cooking_tools.navigate_steps("session-1", "jump", step_index=-5)
    print("Negative Index Result:", res)
    
    # Extremely large scaling
    res = await cooking_tools.scale_recipe("session-1", 0.0)
    print("Zero Scaling Result:", res)

await test_error_handling()

Negative Index Result: {'status': 'success', 'current_step': -5}
Zero Scaling Result: {'status': 'success', 'new_multiplier': 0.0}


## 4. Multi-Tool Composition
Executing a sequence of tools based on a single intent.

In [5]:
async def multi_tool_compose(session_id):
    # Scenario: Double recipe and set a timer for the first step
    scaling = await cooking_tools.scale_recipe(session_id, 2.0)
    timer = await cooking_tools.set_kitchen_timer(session_id, 1800, "Slow Cook")
    nav = await cooking_tools.navigate_steps(session_id, "next")
    
    return {"scaling": scaling, "timer": timer, "navigation": nav}

print("Composition Result:", await multi_tool_compose("session-compose"))

Composition Result: {'scaling': {'status': 'success', 'new_multiplier': 2.0}, 'timer': {'status': 'success', 'timer_id': 'd2d67ab4', 'label': 'Slow Cook', 'duration': 1800}, 'navigation': {'status': 'success', 'current_step': 1}}
